# Colab template — trening i ewaluacja (ADM / TGN)

Szablon sesji Google Colab dla projektu **adm-project-tgnn**.

Ten notebook:
1. montuje Google Drive,
2. instaluje repozytorium i zależności,
3. sprawdza ścieżki na Drive (`data/processed.zip`),
4. rozpakowuje dane lokalnie w sesji Colab (szybszy I/O),
5. loguje do Weights & Biases i weryfikuje `entity` / `project`,
6. drukuje podsumowanie sesji gotowej do treningu.

**Wymagania na Drive:**
```text
<drive_project>/
├── data/processed.zip
└── checkpoints/          # tworzone automatycznie przy treningu
```

Stałe W&B są w kodzie: `src/config/wandb_settings.py` (`entity=project-nn`, `project=adm-project-tgnn`).

## 0. Konfiguracja użytkownika

Jedyna sekcja do ręcznej edycji przed uruchomieniem.

In [ ]:
from pathlib import Path

# Ścieżka do folderu projektu na Google Drive (zawiera data/processed.zip)
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/adm-project-tgnn")

# Lokalny katalog repozytorium w sesji Colab
REPO_DIR = Path("/content/adm-project-tgnn")

# Ustaw URL przy pierwszym uruchomieniu (None = zakładam, że repo już jest w REPO_DIR)
REPO_URL = None  # np. "https://github.com/<org>/adm-project-tgnn.git"

# Wariant danych z preprocessingu
PROCESSED_VARIANT = "subsample_1_32_clicks_only"

# Nazwa runa w wandb i podkatalog checkpointów na Drive
RUN_NAME = "colab-template-smoke"

## 1. Repozytorium i zależności

In [ ]:
import subprocess
import sys

if REPO_URL and not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if not REPO_DIR.is_dir():
    raise FileNotFoundError(
        f"Repo not found at {REPO_DIR}. Set REPO_URL or clone the project manually."
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

print(f"Repo ready: {REPO_DIR}")

## 2. Google Drive — mount, walidacja ścieżek, rozpakowanie danych

In [ ]:
from pprint import pprint

from src.colab.setup import check_drive_layout, prepare_colab_session
from src.config.train_config import TrainConfig

layout = check_drive_layout(DRIVE_PROJECT_DIR)
print("Drive layout check:")
pprint(
    {
        "drive_project_dir": layout.drive_project_dir,
        "processed_zip": layout.processed_zip,
        "checkpoint_root": layout.checkpoint_root,
        "ok": layout.ok,
    }
)

if not layout.ok:
    raise FileNotFoundError("Drive layout invalid:\n" + "\n".join(layout.errors))

config = TrainConfig.for_colab(
    DRIVE_PROJECT_DIR,
    processed_variant=PROCESSED_VARIANT,
    run_name=RUN_NAME,
    wandb_run_name=RUN_NAME,
)
config = prepare_colab_session(config)

print("\nLocal session paths after unpack:")
pprint(
    {
        "processed_dir": config.processed_dir,
        "meta_path": config.meta_path,
        "checkpoint_dir": config.checkpoint_dir,
    }
)

## 3. Weights & Biases — login i weryfikacja entity / project

In [ ]:
from src.config.wandb_settings import (
    WANDB_ENTITY,
    WANDB_PROJECT,
    expected_wandb_settings,
    verify_wandb_access,
)

expected = expected_wandb_settings()
print(f"Expected wandb entity : {expected.entity}  (module default: {WANDB_ENTITY})")
print(f"Expected wandb project: {expected.project} (module default: {WANDB_PROJECT})")

assert config.wandb_entity == WANDB_ENTITY
assert config.wandb_project == WANDB_PROJECT

wandb_info = verify_wandb_access(expected)
print("\nwandb access OK:")
for key, value in wandb_info.items():
    print(f"  {key}: {value}")

## 4. Smoke test logowania do wandb + podsumowanie danych

Krótki run testowy (`setup/ok=1`) potwierdza, że metryki trafiają do właściwego projektu. Pełny trening uruchomisz później przez `train_gru4rec(config)`.

In [ ]:
import wandb

from src.data.meta import load_meta

meta = load_meta(config.processed_dir)

run = wandb.init(
    project=config.wandb_project,
    entity=config.wandb_entity,
    name=config.wandb_run_name,
    config=config.to_dict(),
    tags=["colab-template", "setup"],
)
wandb.log({"setup/ok": 1})
print(f"wandb run: {run.url}")

print("\nDataset summary (from meta.json):")
pprint(
    {
        "n_items": meta["stats"]["n_items_train_vocab"],
        "train_examples": meta["stats"]["train_examples"],
        "val_examples": meta["stats"]["val_examples"],
        "index_conventions": meta["index_conventions"]["gru4rec"],
    }
)

run.finish()
print("\nSession ready for training. Next step:")
print("  from src.training.train_gru4rec import train_gru4rec")
print("  train_gru4rec(config)")